# Analiza zahtevka stroškov

Ta zvezek prikazuje, kako ustvariti agente, ki uporabljajo vtičnike za obdelavo stroškov potovanj iz lokalnih posnetkov računov, ustvarjanje e-pošte za zahtevek stroškov in vizualizacijo podatkov o stroških s tortnim diagramom. Agenti dinamično izbirajo funkcije glede na kontekst naloge.

Koraki:
1. OCR agent obdela sliko lokalnega računa in izvleče podatke o potnih stroških.
2. Email agent pripravi e-pošto za zahtevek stroškov.

### Primer scenarija potnih stroškov:
Predstavljajte si, da ste zaposleni na službenem potovanju za poslovni sestanek v drugem mestu. Vaše podjetje ima politiko povračila vseh razumnim stroškov, povezanih s potovanjem. Tukaj je pregled morebitnih potnih stroškov:
- Prevoz:
Letalska vozovnica za povratno pot iz vašega domačega mesta do kraja destinacije.
Taksi ali storitve prevoza na klic do in iz letališča.
Lokalni prevoz v mestu destinacije (kot so javni prevoz, najem avtomobilov ali taksiji).

- Nastanitev:
Hotel za tri noči v srednje kakovostnem poslovnem hotelu blizu prizorišča sestanka.

- Obroki:
Dnevni dodatek za obroke za zajtrk, kosilo in večerjo, glede na politiko dnevnic podjetja.

- Razni stroški:
Parkirnine na letališču.
Stroški dostopa do interneta v hotelu.
Napitnine ali manjši stroški storitev.

- Dokumentacija:
Predložite vse račune (letenje, taksiji, hotel, obroki itd.) in izpolnjen obrazec stroškov za povračilo.


## Uvozi zahtevane knjižnice

Uvozi potrebne knjižnice in module za zvezek.


In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import base64
import dotenv
from typing import Annotated, List

from pydantic import BaseModel, Field

from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

 ## Določite modele stroškov

 Ustvarite Pydantic model za posamezne stroške in razred ExpenseFormatter za pretvorbo uporabniške poizvedbe v strukturirane podatke o stroških.

 Vsak strošek bo predstavljen v formatu:
 `{'date': '07-Mar-2025', 'description': 'let do cilja', 'amount': 675.99, 'category': 'Prevoz'}`


In [ ]:
class Expense(BaseModel):
    date: str = Field(..., description="Date of expense in dd-MMM-yyyy format")
    description: str = Field(..., description="Expense description")
    amount: float = Field(..., description="Expense amount")
    category: str = Field(..., description="Expense category (e.g., Transportation, Meals, Accommodation, Miscellaneous)")

class ExpenseFormatter(BaseModel):
    raw_query: str = Field(..., description="Raw query input containing expense details")
    
    def parse_expenses(self) -> List[Expense]:
        """
        Parses the raw query into a list of Expense objects.
        Expected format: "date|description|amount|category" separated by semicolons.
        """
        expense_list = []
        for expense_str in self.raw_query.split(";"):
            if expense_str.strip():
                parts = expense_str.strip().split("|")
                if len(parts) == 4:
                    date, description, amount, category = parts
                    try:
                        expense = Expense(
                            date=date.strip(),
                            description=description.strip(),
                            amount=float(amount.strip()),
                            category=category.strip()
                        )
                        expense_list.append(expense)
                    except ValueError as e:
                        print(f"[LOG] Parse Error: Invalid data in '{expense_str}': {e}")
        return expense_list

## Določanje orodij - Generiranje e-pošte

Ustvarite funkcijo orodja za generiranje e-pošte za oddajo zahtevka za povračilo stroškov.
- To orodje uporablja dekorator `@tool` iz Microsoft Agent Framework.
- Izračuna skupni znesek stroškov in podrobnosti oblikuje v telo e-pošte.


In [ ]:
@tool(approval_mode="never_require")
def generate_expense_email(
    expense_data: Annotated[str, "Semicolon-separated expense entries in 'date|description|amount|category' format"]
) -> str:
    """Generate an email to submit an expense claim to the Finance Team."""
    formatter = ExpenseFormatter(raw_query=expense_data)
    expenses = formatter.parse_expenses()
    if not expenses:
        return "No valid expenses found to include in the email."
    total_amount = sum(e.amount for e in expenses)
    email_body = "Dear Finance Team,\n\n"
    email_body += "Please find below the details of my expense claim:\n\n"
    for e in expenses:
        email_body += f"- {e.date} | {e.description}: ${e.amount:.2f} ({e.category})\n"
    email_body += f"\nTotal Amount: ${total_amount:.2f}\n\n"
    email_body += "Receipts for all expenses are attached for your reference.\n\n"
    email_body += "Thank you,\n[Your Name]"
    return email_body

# Orodje za Izvlečenje Stroškov Potovanj iz Slik Računov

Ustvarite funkcijo orodja za izvlečenje stroškov potovanj iz slik računov.
- To orodje uporablja dekorator `@tool` iz Microsoft Agent Framework.
- Prebere sliko računa, jo pretvori v base64 in vrne podatkovni URI za analizo agenta.


In [ ]:
@tool(approval_mode="never_require")
def load_receipt_image(
    image_path: Annotated[str, "Path to the receipt image file"] = "receipt.jpg"
) -> str:
    """Load a receipt image and return its base64-encoded data URI for OCR extraction."""
    try:
        with open(image_path, "rb") as f:
            image_data = base64.b64encode(f.read()).decode("utf-8")
        return f"data:image/jpeg;base64,{image_data}"
    except Exception as e:
        error_msg = f"[LOG] Error loading image '{image_path}': {str(e)}"
        print(error_msg)
        return error_msg

## Obdelava stroškov

Določite agente in jih povežite v zaporedno delovno tok z uporabo `WorkflowBuilder`.
- OCR agent izvleče strukturirane podatke o stroških iz slike računa z uporabo orodja `load_receipt_image`.
- Email agent prejme izvlečene podatke in ustvari profesionalno elektronsko sporočilo za uveljavljanje stroškov z orodjem `generate_expense_email`.
- `WorkflowBuilder` z `add_edge` ustvari zaporedno cevovod: OCR Agent → Email Agent.


In [ ]:
ocr_agent = client.as_agent(
    tools=[load_receipt_image],
    name="OCRAgent",
    instructions=(
        "You are an expert OCR assistant specialized in extracting structured data from receipt images. "
        "Use the 'load_receipt_image' tool to load the receipt image, then analyze it and extract "
        "travel-related expense details in the format: 'date|description|amount|category' separated by semicolons. "
        "Follow these rules: "
        "- Date: Convert dates (e.g., '4/4/22') to 'dd-MMM-yyyy' (e.g., '04-Apr-2022'). "
        "- Description: Extract item names. "
        "- Amount: Use numeric values (e.g., '4.50' from '$4.50'). "
        "- Category: Infer from context (e.g., 'Meals' for food, 'Transportation' for travel, "
        "'Accommodation' for lodging, 'Miscellaneous' otherwise). "
        "Ignore totals, subtotals, or service charges unless they are itemized expenses. "
        "If no expenses are found, return 'No expenses detected'. "
        "Return only the structured data, no additional text."
    ),
)

email_agent = client.as_agent(
    name="EmailAgent",
    tools=[generate_expense_email],
    instructions=(
        "You are an expense claim email generator. Take the travel expense data from the previous agent "
        "(in 'date|description|amount|category' format separated by semicolons) and use the "
        "'generate_expense_email' tool to produce a professional expense claim email. "
        "Pass the semicolon-separated expense data directly to the tool."
    ),
)

## Glavna funkcija

Ustvari zaporedno potek dela in jo zaženi za obdelavo slike računa ter ustvarjanje elektronske pošte za uveljavljanje stroškov.


> **Opomba:** Ta potek dela trenutno posreduje sliko računa kot besedilo base64, ki ga večina klepetalnih modelov (vključno z gpt-4.1-mini) ne bo obravnavala kot sliko.
> Lahko tudi preseže okno konteksta modela. Priporočljivo je izvajati OCR z Azure AI Vision (ali drugim OCR orodjem) in posredovati samo izločeno besedilo ali pa preurediti, da se slika pošlje kot sporočilo `image_url`.
> Če želite samo preprečiti napake v kontekstu, uporabite manjšo sliko računa ali model z večjim oknom konteksta.


In [ ]:
workflow = WorkflowBuilder(start_executor=ocr_agent) \
    .add_edge(ocr_agent, email_agent) \
    .build()

prompt = (
    "Please extract the raw text from the receipt image at 'receipt.jpg', "
    "focusing on travel expenses like dates, descriptions, amounts, and categories "
    "(e.g., Transportation, Accommodation, Meals, Miscellaneous). "
    "Then generate a professional expense claim email."
)

last_author = None
events = workflow.run(
    prompt,
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"# Agent - {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Omejitev odgovornosti**:
Ta dokument je bil preveden z uporabo AI prevajalske storitve [Co-op Translator](https://github.com/Azure/co-op-translator). Čeprav si prizadevamo za natančnost, vas prosimo, da upoštevate, da avtomatizirani prevodi lahko vsebujejo napake ali netočnosti. Izvirni dokument v njegovem izvirnem jeziku je treba obravnavati kot avtoritativni vir. Za kritične informacije je priporočljiv strokovni človeški prevod. Ne odgovarjamo za morebitna nesporazume ali napačne interpretacije, ki izhajajo iz uporabe tega prevoda.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
